# IMD-3 Global Prototype

**Chemin 1** du plan d'extension de l'IMD au catalogue mondial : remplacer le composite à 4 dimensions (M + I + S + T) par une variante à 3 dimensions (M + I + T) qui supprime le facteur sécurité (S) parce qu'aucun équivalent BAAC unifié n'existe à l'échelle mondiale (FARS US, STATS19 UK, CARE EU, BRA BE, ... ont des définitions et géocodages incompatibles).

## Définition de l'IMD-3

L'IMD-3 par station est :

$$\text{IMD-3}_i = w_M \cdot M_i + w_I \cdot I_i + w_T \cdot T_i$$

où les composants $M_i, I_i, T_i \in [0, 1]$ sont des min-max normalisés sur le corpus étudié et les poids sont re-normalisés depuis l'IMD-4 français :

| Composant | Poids IMD-4 | Poids IMD-3 (re-normalisé) |
|:---|---:|---:|
| M multimodalité | 0.578 | **0.642** |
| I infrastructure cyclable | 0.184 | **0.204** |
| T topographie | 0.096 | **0.107** |
| ~~S sécurité~~ | ~~0.142~~ | ~~retiré~~ |

Renormalisation : $w_k^{IMD3} = w_k^{IMD4} / (w_M + w_I + w_T)$.

## Sources globales utilisées

| Composant | Source FR (IMD-4) | Source mondiale (IMD-3) | Statut |
|:---|:---|:---|:---:|
| T | SRTM 30 m + BD ALTI | Open-Elevation API (SRTM 30 m mondial) | OK |
| I | BD TOPO (IGN) | OpenStreetMap Overpass `highway=cycleway` | OK |
| M | GTFS feeds FR | OSM `railway=station` + `station=subway/tram` (proxy) | OK approximé |

## Périmètre du prototype

Pour rester reproductible en moins de 10 minutes, le notebook calcule l'IMD-3 sur **3 systèmes étrangers** déjà fetchés par les pilotes E5 :

- Bicing Barcelona (ES) : 544 stations
- Oslo Bysykkel (NO) : 266 stations
- Bergen Bysykkel (NO) : 97 stations

Pour chaque système, on tire un **échantillon stratifié de 20 stations** afin de limiter les appels Overpass. Une section finale décrit la procédure de mise à l'échelle pour les ~640 systèmes mondiaux avec données.

## 1. Imports et configuration

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import requests

TMP = Path(os.environ.get("TMPDIR") or os.environ.get("TEMP") or "/tmp")
HERE = Path.cwd()
CACHE = HERE / "_cache"
CACHE.mkdir(exist_ok=True)

# Re-normalised IMD-3 weights
W_M = 0.578 / (0.578 + 0.184 + 0.096)
W_I = 0.184 / (0.578 + 0.184 + 0.096)
W_T = 0.096 / (0.578 + 0.184 + 0.096)
print(f"IMD-3 weights -> M={W_M:.3f} I={W_I:.3f} T={W_T:.3f} (sum={W_M+W_I+W_T:.3f})")

# API endpoints (no auth needed)
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
ELEV_URL     = "https://api.open-elevation.com/api/v1/lookup"

# Sampling parameters
SAMPLE_PER_CITY = 20         # number of stations per system
BUFFER_RADIUS_M = 300         # buffer for cycleway + transit queries
SEED = 42
rng = np.random.default_rng(SEED)

# Polite-client delay between API calls (respect Overpass usage policy)
OVERPASS_DELAY_S = 1.2
ELEV_DELAY_S     = 0.3

## 2. Charger les stations des 3 systèmes pilotes

On lit les payloads `station_information` déjà fetchés par les pilotes E5 (Bicing, Oslo, Bergen). Si les fichiers n'existent pas dans `$TEMP`, on les re-fetch.

In [ ]:
FEEDS = {
    "Bicing Barcelona": {
        "country": "ES",
        "local_path": TMP / "bicing_stations.json",
        "url": "https://barcelona.publicbikesystem.net/customer/gbfs/v2/en/station_information",
    },
    "Oslo Bysykkel": {
        "country": "NO",
        "local_path": TMP / "oslo_stations.json",
        "url": "https://gbfs.urbansharing.com/oslobysykkel.no/station_information.json",
    },
    "Bergen Bysykkel": {
        "country": "NO",
        "local_path": TMP / "bergen_stations.json",
        "url": "https://gbfs.urbansharing.com/bergenbysykkel.no/station_information.json",
    },
}


def load_stations(name: str, meta: dict) -> pd.DataFrame:
    """Load station_information from local cache, or fetch if missing."""
    p = meta["local_path"]
    if not p.exists():
        print(f"  ... {name}: fetching from {meta['url']}")
        r = requests.get(meta["url"], timeout=15)
        r.raise_for_status()
        p.write_text(r.text, encoding="utf-8")
    data = json.loads(p.read_text(encoding="utf-8"))
    stations = data["data"]["stations"]
    df = pd.DataFrame(stations)
    df["system"] = name
    df["country"] = meta["country"]
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    return df[["system", "country", "station_id", "lat", "lon"]].dropna()


frames = []
for name, meta in FEEDS.items():
    try:
        df_one = load_stations(name, meta)
        print(f"  {name}: {len(df_one):>4} stations loaded")
        frames.append(df_one)
    except Exception as e:
        print(f"  {name}: FAILED ({type(e).__name__})")

stations = pd.concat(frames, ignore_index=True)
print(f"\nTotal stations across pilot systems: {len(stations)}")

## 3. Échantillonnage stratifié

Pour le prototype, on prend 20 stations aléatoires par système (seed = 42). Pour le passage à l'échelle, on remplace cette cellule par `sample = stations` (toutes les stations) et on traite par batches.

In [ ]:
def stratified_sample(df: pd.DataFrame, k_per_group: int) -> pd.DataFrame:
    """Sample k stations per system, deterministic via the global seed."""
    return (
        df.groupby("system", group_keys=False)
        .apply(lambda g: g.sample(
            n=min(k_per_group, len(g)), random_state=SEED, replace=False,
        ))
        .reset_index(drop=True)
    )


sample = stratified_sample(stations, SAMPLE_PER_CITY)
print(f"Sampled {len(sample)} stations (per system: {SAMPLE_PER_CITY})")
sample.groupby("system").size()

## 4. Composante T - Topographie

Open-Elevation expose une API publique qui retourne l'altitude SRTM en mètres pour une liste de points. On batche par 100 points par requête.

In [ ]:
def fetch_elevations(latlon_pairs: list[tuple[float, float]]) -> list[float]:
    """Open-Elevation batch lookup. Returns elevation in metres, NaN on failure."""
    out: list[float] = []
    BATCH = 50  # keep below Open-Elevation's per-request limit
    for i in range(0, len(latlon_pairs), BATCH):
        chunk = latlon_pairs[i:i + BATCH]
        payload = {"locations": [{"latitude": la, "longitude": lo} for la, lo in chunk]}
        try:
            r = requests.post(ELEV_URL, json=payload, timeout=30)
            r.raise_for_status()
            elev = [pt.get("elevation") for pt in r.json().get("results", [])]
        except Exception as e:
            print(f"  Open-Elevation batch {i}: FAILED ({type(e).__name__})")
            elev = [np.nan] * len(chunk)
        out.extend(elev)
        time.sleep(ELEV_DELAY_S)
    return out


elev_cache = CACHE / "elevations.csv"
if elev_cache.exists():
    elev_df = pd.read_csv(elev_cache)
    print(f"  Loaded {len(elev_df)} elevations from cache")
else:
    print(f"  Fetching {len(sample)} elevations from Open-Elevation...")
    pairs = list(zip(sample["lat"], sample["lon"]))
    elev = fetch_elevations(pairs)
    elev_df = sample[["station_id", "lat", "lon"]].copy()
    elev_df["elevation_m"] = elev
    elev_df.to_csv(elev_cache, index=False)

sample = sample.merge(elev_df[["station_id", "elevation_m"]], on="station_id", how="left")
print(sample.groupby("system")["elevation_m"].agg(["mean", "std", "min", "max"]).round(1))

## 5. Composante I - Infrastructure cyclable

Requête Overpass : longueur cumulée de `way[highway=cycleway]` dans un buffer de 300 m autour de chaque station. Ce composant approxime `infra_cyclable_km` du Gold Standard FR (qui s'appuie sur BD TOPO).

**Attention :** chaque station déclenche une requête Overpass ; budget temps `n_stations * 1.2 s`. Pour 60 stations c'est `~72 s`.

In [ ]:
def overpass_cycleway_length_m(lat: float, lon: float, radius: int = BUFFER_RADIUS_M) -> float:
    """Total length of OSM cycleway ways within `radius` m of (lat, lon).

    Uses a haversine-on-segments approximation (cheap, accurate within 1% at 300 m).
    Returns 0.0 on failure or when no cycleway is found.
    """
    q = f"""
    [out:json][timeout:25];
    (
      way["highway"="cycleway"](around:{radius},{lat},{lon});
      way["cycleway"](around:{radius},{lat},{lon});
      way["cycleway:left"](around:{radius},{lat},{lon});
      way["cycleway:right"](around:{radius},{lat},{lon});
      way["cycleway:both"](around:{radius},{lat},{lon});
    );
    out geom;
    """
    try:
        r = requests.post(OVERPASS_URL, data={"data": q}, timeout=35)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return 0.0

    total = 0.0
    for el in data.get("elements", []):
        geom = el.get("geometry") or []
        for a, b in zip(geom, geom[1:]):
            total += _haversine_m(a["lat"], a["lon"], b["lat"], b["lon"])
    return total


def _haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6_371_000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


infra_cache = CACHE / "cycleway_m.csv"
if infra_cache.exists():
    infra_df = pd.read_csv(infra_cache)
    print(f"  Loaded {len(infra_df)} cycleway measurements from cache")
else:
    print(f"  Querying Overpass for {len(sample)} stations ({BUFFER_RADIUS_M} m radius)...")
    results = []
    for i, row in sample.iterrows():
        m = overpass_cycleway_length_m(row["lat"], row["lon"])
        results.append(m)
        if (i + 1) % 10 == 0:
            print(f"    {i+1}/{len(sample)} done")
        time.sleep(OVERPASS_DELAY_S)
    infra_df = sample[["station_id"]].copy()
    infra_df["cycleway_m"] = results
    infra_df.to_csv(infra_cache, index=False)

sample = sample.merge(infra_df, on="station_id", how="left")
print(sample.groupby("system")["cycleway_m"].agg(["mean", "median", "max"]).round(0))

## 6. Composante M - Multimodalité

Le Gold Standard FR utilise les feeds GTFS français pour compter `gtfs_heavy_stops_300m`. Pour le passage au monde, on utilise un proxy OSM :

- `railway=station` + `subway` + `tram_stop` + `light_rail` dans un buffer de 300 m

Cette approximation suit la même intention sémantique (compter les arrêts lourds de transport en commun à proximité) sans dépendre des feeds GTFS pays-par-pays.

In [ ]:
def overpass_heavy_transit_count(lat: float, lon: float, radius: int = BUFFER_RADIUS_M) -> int:
    """Count OSM heavy-transit stops within `radius` m: station, subway, tram, light_rail."""
    q = f"""
    [out:json][timeout:25];
    (
      node["railway"="station"](around:{radius},{lat},{lon});
      node["railway"="subway_entrance"](around:{radius},{lat},{lon});
      node["railway"="tram_stop"](around:{radius},{lat},{lon});
      node["public_transport"="station"](around:{radius},{lat},{lon});
      node["station"="subway"](around:{radius},{lat},{lon});
      node["station"="light_rail"](around:{radius},{lat},{lon});
    );
    out count;
    """
    try:
        r = requests.post(OVERPASS_URL, data={"data": q}, timeout=35)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return 0
    elements = data.get("elements", [])
    if elements and "tags" in elements[0]:
        return int(elements[0]["tags"].get("nodes", 0))
    return len([e for e in elements if e.get("type") == "node"])


transit_cache = CACHE / "transit_count.csv"
if transit_cache.exists():
    transit_df = pd.read_csv(transit_cache)
    print(f"  Loaded {len(transit_df)} transit counts from cache")
else:
    print(f"  Querying Overpass for {len(sample)} stations (heavy transit, {BUFFER_RADIUS_M} m)...")
    results = []
    for i, row in sample.iterrows():
        c = overpass_heavy_transit_count(row["lat"], row["lon"])
        results.append(c)
        if (i + 1) % 10 == 0:
            print(f"    {i+1}/{len(sample)} done")
        time.sleep(OVERPASS_DELAY_S)
    transit_df = sample[["station_id"]].copy()
    transit_df["transit_count"] = results
    transit_df.to_csv(transit_cache, index=False)

sample = sample.merge(transit_df, on="station_id", how="left")
print(sample.groupby("system")["transit_count"].agg(["mean", "median", "max"]).round(2))

## 7. Normalisation et calcul de l'IMD-3

Min-max sur chaque composant à l'échelle du corpus prototype (3 systèmes). En production, normaliser sur le corpus complet (toutes les villes mondiales auditées).

Pour la topographie, l'IMD-FR pénalise les fortes pentes : on utilise donc `T = 1 - normalize(roughness)` où la roughness est l'écart-type local. Pour simplifier le prototype, on prend `T = 1 - normalize(elevation_above_min)` qui capture l'idée "plus on est haut par rapport au minimum de la ville, plus c'est dur à vélo".

Les composants I et M sont "plus c'est mieux", donc normalisation directe.

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    return (s - lo) / (hi - lo)


# Per-station topography component: inversed elevation gap to city minimum
sample["elev_gap"] = sample.groupby("system")["elevation_m"].transform(
    lambda s: s - s.min()
)
sample["T_i"] = 1.0 - minmax(sample["elev_gap"])

sample["I_i"] = minmax(sample["cycleway_m"])
sample["M_i"] = minmax(sample["transit_count"].astype(float))
sample["IMD3_i"] = W_M * sample["M_i"] + W_I * sample["I_i"] + W_T * sample["T_i"]

sample[["system", "station_id", "T_i", "I_i", "M_i", "IMD3_i"]].head(8)

## 8. Agrégation par ville et comparaison

On agrège l'IMD-3 par système (médiane robustifie aux outliers de l'échantillon). Une comparaison indicative avec les villes françaises de référence est ajoutée.

In [ ]:
by_city = sample.groupby(["system", "country"]).agg(
    n_stations_sample=("station_id", "size"),
    mean_elevation_m=("elevation_m", "mean"),
    mean_cycleway_m=("cycleway_m", "mean"),
    mean_transit=("transit_count", "mean"),
    IMD3_median=("IMD3_i", "median"),
    IMD3_mean=("IMD3_i", "mean"),
).reset_index()

# Add reference FR cities for context (numbers from companion IMD paper)
fr_reference = pd.DataFrame({
    "system": ["Strasbourg (Vélhop)", "Montpellier (Vélomagg)", "Paris (Vélib')", "Mulhouse"],
    "country": ["FR"] * 4,
    "n_stations_sample": [None] * 4,
    "mean_elevation_m": [None] * 4,
    "mean_cycleway_m": [None] * 4,
    "mean_transit": [None] * 4,
    "IMD3_median": [None] * 4,
    "IMD3_mean": [None] * 4,
    "IMD_FR_reference": [94.1, 86.8, 67.1, 75.1],
})

out = pd.concat([by_city, fr_reference], ignore_index=True)
out

## 9. Lecture des résultats

Le tableau ci-dessus expose les IMD-3 prototype pour les 3 villes étrangères, accompagnés à titre comparatif des IMD-FR des 4 villes-référence du papier compagnon. Les IMD-3 prototype sont sur l'échelle `[0, 1]` ; pour les rapporter à l'échelle 0-100 utilisée par le papier FR, multiplier par 100.

**Limites importantes du prototype :**

1. La normalisation min-max sur 3 villes seulement gonfle artificiellement les écarts. À l'échelle ~640 villes, les valeurs convergent vers une distribution plus représentative.
2. L'échantillonnage à 20 stations par ville sous-estime la variance intra-ville. Production : toutes les stations.
3. Le composant M (transit OSM) est un proxy approximé de GTFS heavy stops. Pour une comparaison stricte avec l'IMD-FR, il faut consommer les GTFS feeds mondiaux via MobilityData.
4. Le composant T est une approximation "hauteur au-dessus du minimum de la ville" plutôt que l'indice de rugosité multi-échelle utilisé en FR (BD ALTI + dérivées du gradient).
5. Pas de validation croisée vis-à-vis de l'IMD-FR sur les villes communes : les composants ne sont pas calibrés sur le même référentiel.

## 10. Passage à l'échelle complète

Pour étendre ce prototype aux ~640 systèmes mondiaux dock-based avec données :

### Étape A. Récupérer les coordonnées de toutes les stations

Étendre `papers/01_gold_standard/experiments/e5_europe/massive_audit.py` pour persister `world_stations.parquet` avec `(system_id, station_id, lat, lon)` sur les ~640 systèmes reachables.

Estimation : 1 fetch par système, ~10 min total avec parallélisation à 16 workers (déjà en place dans le pipeline).

### Étape B. Topographie pour ~230 000 stations

Open-Elevation rate-limit : ~0.5 req/s en batches de 50 = ~25 stations/s = **~150 min total**.

Alternative recommandée : télécharger les tuiles SRTM 30 m localement (~10 GB) et lookup local via `rasterio`. Coût : 1 h de téléchargement, lookup instantané.

### Étape C. Infrastructure et transit via Overpass

Overpass-API public rate-limit : ~2 req/s sustained = **~32 h** pour 230 000 stations × 2 requêtes (cycleway + transit).

Trois solutions :

1. **Self-hosted Overpass** : un serveur Overpass local (8 GB de RAM + dump Europe = 30 GB disque) traite la même charge en ~2 h.
2. **Agrégation par ville** : faire l'analyse à l'échelle ville (1 polygon par ville) plutôt que station. Beaucoup moins de requêtes.
3. **Sampling stratifié** : 50 stations par ville suffisent statistiquement pour estimer l'IMD-3 médian (cf. simulations Bootstrap).

### Étape D. Calibrage des poids IMD-3

Les poids re-normalisés (0.642, 0.204, 0.107) supposent que la suppression de S est neutre. C'est une approximation : un ré-optimisation des poids sur le corpus IMD-3 mondial donnerait probablement des coefficients différents (S est corrélé à I via la qualité des infrastructures cyclables).

Procédure recommandée : sur les villes FR du Gold Standard, calculer IMD-3 et IMD-4 puis mesurer la corrélation Spearman. Si $\rho > 0.9$, l'IMD-3 est une approximation raisonnable. Si $\rho < 0.7$, il faut re-optimiser ou changer de méthodologie.

### Étape E. Production papier compagnon

L'IMD-3 mondial mérite un papier compagnon court : *"IMD-3: a 3-dimensional soft-mobility composite for the 640 dock-based BSS worldwide"*. Format Scientific Data ou Transportation Research Part C. Ne pas confondre avec le papier IMD-4 français (qui reste valable pour la France où S est mesurable).

---

*Prototype IMD-3 - R. Fossé et G. Pallares, BikeShare-ICT, CESI LINEACT, 2026.*